# Factorial measurement allocation study

Runs the rank-versus-width calibration and test-stopping comparison on three binary datasets. Each comparison uses paired shot streams. The two evaluation protocols use seeds 50–59 and 40–49.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn", "torch": "torch", "matplotlib": "matplotlib"}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

In [ ]:
%%writefile qcp_rank_v2.py
import argparse, math, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import beta
from sklearn.datasets import load_breast_cancer, load_wine, load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch

OUT = Path(__file__).resolve().parent
STAGES = [8,16,32,64,128,256,512,1024]
ALPHA, BETA = 0.10, 0.01
BETA_CAL, BETA_TEST = 0.005, 0.005


GATE_PAIRS = {}
CX_PERMS = {}
for j in range(4):
    step = 1 << (3-j)
    GATE_PAIRS[j] = (np.array([i for i in range(16) if (i & step) == 0]),
                     np.array([i for i in range(16) if (i & step) == 0])+step)
for c,t in [(0,1),(1,2),(2,3),(3,0),(1,0),(2,0)]:
    CX_PERMS[(c,t)] = torch.tensor([i ^ (1 << (3-t)) if (i & (1 << (3-c))) else i for i in range(16)],dtype=torch.long)
MEASURE1 = torch.tensor([i for i in range(16) if i & 8],dtype=torch.long)

def ry(state, angle, wire):
    idx0,idx1 = GATE_PAIRS[wire]
    a,b = state[:,idx0],state[:,idx1]
    cs = torch.cos(angle/2).reshape(-1,1); sn = torch.sin(angle/2).reshape(-1,1)
    out = state.clone()
    out[:,idx0] = cs*a - sn*b
    out[:,idx1] = sn*a + cs*b
    return out

def model_p(x, theta):
    n = x.shape[0]
    s = torch.zeros((n,16),dtype=x.dtype,device=x.device)
    s[:,0]=1.0
    for j in range(4): s=ry(s,x[:,j],j)
    for c,t in [(0,1),(1,2),(2,3)]: s=s[:,CX_PERMS[(c,t)]]
    for j in range(4): s=ry(s,theta[j].expand(n)+0.45*x[:,j],j)
    for c,t in [(3,0),(2,0),(1,0)]: s=s[:,CX_PERMS[(c,t)]]
    for j in range(4): s=ry(s,theta[j+4].expand(n),j)
    return torch.sum(s[:,MEASURE1]**2,axis=1).clamp(1e-6,1-1e-6)

def prepare(name, seed):
    if name=='breast':
        data=load_breast_cancer(); X=data.data; y=data.target
    elif name=='wine01':
        data=load_wine(); m=data.target<2; X=data.data[m]; y=data.target[m]
    elif name=='digits38':
        data=load_digits(); m=np.isin(data.target,[3,8]); X=data.data[m]; y=(data.target[m]==8).astype(int)
    else: raise ValueError(name)
    ind=np.arange(len(y))
    a,b=train_test_split(ind,test_size=0.5,stratify=y,random_state=seed)
    c,d=train_test_split(b,test_size=0.5,stratify=y[b],random_state=seed+101)
    scaler=StandardScaler().fit(X[a]); xs=scaler.transform(X)
    pca=PCA(n_components=4,random_state=seed).fit(xs[a]); z=pca.transform(xs)
    sc=StandardScaler().fit(z[a]); z=np.pi*0.78*np.tanh(sc.transform(z)/1.8)
    return (z[a],y[a]),(z[c],y[c]),(z[d],y[d])

def train(train_xy, seed, epochs):
    torch.manual_seed(seed)
    X,y=train_xy
    xt=torch.as_tensor(X,dtype=torch.float64)
    yt=torch.as_tensor(y,dtype=torch.float64)
    theta=torch.nn.Parameter(torch.randn(8,dtype=torch.float64)*0.25)
    opt=torch.optim.Adam([theta],lr=0.07)
    bestloss=float('inf'); best=None
    for k in range(epochs):
        opt.zero_grad(); p=model_p(xt,theta)
        loss=-(yt*torch.log(p)+(1-yt)*torch.log(1-p)).mean()+0.0001*(theta**2).mean()
        loss.backward(); opt.step()
        if loss.item()<bestloss: bestloss=loss.item(); best=theta.detach().clone()
    return best

def predict(z, theta):
    with torch.no_grad(): return model_p(torch.tensor(z,dtype=torch.float64),theta).numpy()

def score_interval(p_lo,p_hi,y):
    return np.where(y==1,1-p_hi,p_lo),np.where(y==1,1-p_lo,p_hi)

def cp_interval(successes,t,err):
    successes=np.asarray(successes,dtype=float)
    lo=np.where(successes>0,beta.ppf(err/2, successes,np.maximum(t-successes+1,1)),0.)
    hi=np.where(successes<t,beta.ppf(1-err/2,successes+1,np.maximum(t-successes,1)),1.)
    return np.maximum(0,np.nan_to_num(lo,nan=0)),np.minimum(1,np.nan_to_num(hi,nan=1))

def order_threshold(v,k):
    return 1.0 if k>len(v) else float(np.partition(v,k-1)[k-1])

def calibrate(p,y,rng,alpha=ALPHA):
    n=len(y); k=math.ceil((n+1)*(1-alpha+BETA))
    p_lo=np.zeros(n); p_hi=np.ones(n); successes=np.zeros(n,dtype=int)
    stage=np.zeros(n,dtype=int); stopped=np.zeros(n,dtype=bool)
    e=BETA_CAL/(n*len(STAGES))
    qlo,qhi=0.,1.
    for stage_idx,t in enumerate(STAGES):
        upd=np.where(~stopped)[0]
        if len(upd)==0: break
        delta=t-np.where(stage[upd]>0,np.array(STAGES)[stage[upd]-1],0)
        successes[upd]+=rng.binomial(delta,p[upd]); stage[upd]=stage_idx+1
        l,u=cp_interval(successes[upd],t,e)
        p_lo[upd]=np.maximum(p_lo[upd],l);p_hi[upd]=np.minimum(p_hi[upd],u)
        s_lo,s_hi=score_interval(p_lo,p_hi,y)
        qlo=order_threshold(s_lo,k);qhi=order_threshold(s_hi,k)


        stopped = ((s_hi < qlo) | (s_lo > qhi))

    s_lo,s_hi=score_interval(p_lo,p_hi,y)
    qlo,qhi=order_threshold(s_lo,k),order_threshold(s_hi,k)
    return qlo,qhi,int(sum(np.array(STAGES)[np.maximum(0,stage-1)])),(int((~stopped).sum())),k

def certify_test(p, qlo,qhi,rng):
    cnt=0; prev=0; plo=0.;phi=1.;nshots=STAGES[-1]
    e=BETA_TEST/len(STAGES)
    for t in STAGES:
        cnt+=rng.binomial(t-prev,p);prev=t
        l,u=cp_interval(cnt,t,e)
        plo=max(plo,float(l));phi=min(phi,float(u))

        a_lo=np.array([plo,1-phi]);a_hi=np.array([phi,1-plo])
        status=np.where(a_hi<=qlo,1,np.where(a_lo>qhi,-1,0))
        if np.all(status!=0): nshots=t;break

    set_y=np.array([int(plo<=qhi),int(1-phi<=qhi)],dtype=int)
    resolved=int(np.all(status!=0))
    return set_y,nshots,resolved

def pset_from_scores(prob,q):
    return np.column_stack([prob<=q,1-prob<=q])


CAL_EARLY_FRAC = 0.45
TEST_EARLY_FRAC = 0.05
METHODS = ['V1_adaptive','V2_rank_upper','V2_endpoint','fixed_certified_256',
           'fixed_certified_512','fixed_certified_1024','standard_QCP_64','oracle_ideal']


def sample_prefixes(probs, rng):


    probs=np.asarray(probs)
    return np.cumsum((rng.random((len(probs),1024)) < probs[:,None]).astype(np.int16),axis=1)


def one_sided_upper(count, t, err):

    c=np.asarray(count)
    return np.where(c<t,beta.ppf(1-err,c+1,np.maximum(t-c,1)),1.)


def update_nested(lo_prev, hi_prev, candidate_lo, candidate_hi):


    lo_next=np.maximum(lo_prev,candidate_lo)
    hi_next=np.minimum(hi_prev,candidate_hi)
    conflict=lo_next>hi_next
    lo_next=np.where(conflict,lo_prev,lo_next)
    hi_next=np.where(conflict,hi_prev,hi_next)
    return lo_next,hi_next

def old_calibration(y, shots):

    n=len(y); k=math.ceil((n+1)*(1-ALPHA+BETA))
    pl=np.zeros(n);pu=np.ones(n)
    stage=np.zeros(n,dtype=int);active=np.ones(n,dtype=bool)
    err=BETA_CAL/(n*len(STAGES))
    for j,t in enumerate(STAGES):
        idx=np.flatnonzero(active)
        if not len(idx):break
        lo,hi=cp_interval(shots[idx,t-1],t,err)
        pl[idx],pu[idx]=update_nested(pl[idx],pu[idx],lo,hi)
        stage[idx]=j+1
        s_l,s_u=score_interval(pl,pu,y)
        qlo=order_threshold(s_l,k);qhi=order_threshold(s_u,k)
        active=~((s_u<qlo)|(s_l>qhi))
    s_l,s_u=score_interval(pl,pu,y)
    return dict(qlo=order_threshold(s_l,k),qhi=order_threshold(s_u,k),
        stage=stage, shots=int(np.sum(np.array(STAGES)[stage-1])),k=k)


def quantile_focused_upper(y,shots,stages,earlyfrac=CAL_EARLY_FRAC):


    n=len(y);k=math.ceil((n+1)*(1-ALPHA+BETA))
    nstages=len(STAGES); upper=np.ones(n)
    for j,t in enumerate(STAGES):
        ix=np.flatnonzero(stages>=j+1)
        if not len(ix):continue
        c=shots[ix,t-1]; score_success=np.where(y[ix]==0,c,t-c)
        err=BETA_CAL/n*(earlyfrac/(nstages-1) if j<nstages-1 else 1-earlyfrac)
        upper[ix]=np.minimum(upper[ix],one_sided_upper(score_success,t,err))
    return order_threshold(upper,k),upper


def test_v1(qlo,qhi,stream):
    n=len(stream);lo=np.zeros(n);hi=np.ones(n)
    stages=np.zeros(n,dtype=int);active=np.ones(n,dtype=bool)
    for j,t in enumerate(STAGES):
        ix=np.flatnonzero(active)
        if not len(ix):break
        ll,uu=cp_interval(stream[ix,t-1],t,BETA_TEST/len(STAGES))
        lo[ix],hi[ix]=update_nested(lo[ix],hi[ix],ll,uu)
        stages[ix]=j+1
        sl=np.column_stack([lo[ix],1-hi[ix]])
        su=np.column_stack([hi[ix],1-lo[ix]])
        active[ix[np.all((su<=qlo)|(sl>qhi),axis=1)]]=False
    sets=np.column_stack([lo<=qhi,1-hi<=qhi])
    return sets,int(np.sum(np.array(STAGES)[stages-1])),stages


def test_v2(qhi,stream,endpoint=False):


    n=len(stream);lo=np.zeros(n);hi=np.ones(n)
    stage=np.zeros(n,dtype=int);active=np.ones(n,dtype=bool)
    if endpoint:
        weights=np.repeat(TEST_EARLY_FRAC/(len(STAGES)-1),len(STAGES))
        weights[-1]=1-TEST_EARLY_FRAC
    else:weights=np.ones(len(STAGES))/len(STAGES)
    for j,t in enumerate(STAGES):
        ix=np.flatnonzero(active)
        if not len(ix):break
        ll,uu=cp_interval(stream[ix,t-1],t,BETA_TEST*weights[j])
        lo[ix],hi[ix]=update_nested(lo[ix],hi[ix],ll,uu)
        stage[ix]=j+1
        sl=np.column_stack([lo[ix],1-hi[ix]])
        su=np.column_stack([hi[ix],1-lo[ix]])
        active[ix[np.all((su<=qhi)|(sl>qhi),axis=1)]]=False
    sets=np.column_stack([lo<=qhi,1-hi<=qhi])
    return sets,int(np.sum(np.array(STAGES)[stage-1])),stage


def fixed_certificate(y,cal_stream,test_stream,t):
    n=len(y);k=math.ceil((n+1)*(1-ALPHA+BETA))
    pcl,pch=cp_interval(cal_stream[:,t-1],t,BETA_CAL/n)
    _,su=score_interval(pcl,pch,y)
    qhi=order_threshold(su,k)
    ptl,pth=cp_interval(test_stream[:,t-1],t,BETA_TEST)
    sets=np.column_stack([ptl<=qhi,1-pth<=qhi])
    return sets,int((n+len(test_stream))*t),qhi


def ordinary_qcp(y,cal_stream,test_stream,t=64):
    n=len(y);k=math.ceil((n+1)*(1-ALPHA))
    pc=cal_stream[:,t-1]/t
    qs=order_threshold(np.where(y==0,pc,1-pc),k)
    pt=test_stream[:,t-1]/t
    return np.column_stack([pt<=qs,1-pt<=qs]),int((n+len(test_stream))*t),qs


def evaluate_one(dataset,seed,epochs=130):
    (zt,yt),(zc,yc),(zv,yv)=prepare(dataset,seed)
    theta=train((zt,yt),seed,epochs)
    pc,pv=predict(zc,theta),predict(zv,theta)

    rng=np.random.default_rng(4321+seed*1009+len(dataset)*113)
    sc=sample_prefixes(pc,rng);sv=sample_prefixes(pv,rng)
    old=old_calibration(yc,sc)
    newq,_=quantile_focused_upper(yc,sc,old['stage'])
    if newq>old['qhi']+1e-10:
        raise AssertionError('V2 upper quantile must be <= V1 on shared outcomes')
    oracleq=order_threshold(np.where(yc==0,pc,1-pc),old['k'])
    oracle=np.column_stack([pv<=oracleq,1-pv<=oracleq])
    v1,s1,g1=test_v1(old['qlo'],old['qhi'],sv)
    v2a,s2a,g2a=test_v2(newq,sv,False)
    v2b,s2b,g2b=test_v2(newq,sv,True)
    if not np.all((~v2a)|v1):
        raise AssertionError('Pathwise subset property V2_rank_upper subset V1 violated')
    methods={
      'V1_adaptive':(v1,old['shots']+s1),
      'V2_rank_upper':(v2a,old['shots']+s2a),
      'V2_endpoint':(v2b,old['shots']+s2b),
      'oracle_ideal':(oracle,np.nan)
    }
    qfix={}
    for t in [256,512,1024]:
        s,c,q=fixed_certificate(yc,sc,sv,t)
        methods[f'fixed_certified_{t}']=(s,c);qfix[t]=q
    s,c,q=ordinary_qcp(yc,sc,sv)
    methods['standard_QCP_64']=(s,c)
    endpoint_match_fraction=float(np.mean(np.all(v2b==methods['fixed_certified_1024'][0],axis=1)))

    rows=[];ti=np.arange(len(yv))
    for name in METHODS:
        s,spent=methods[name]
        rows.append(dict(dataset=dataset,seed=seed,method=name,n_train=len(yt),n_cal=len(yc),n_test=len(yv),
          mean_set_size=float(np.mean(s.sum(axis=1))),coverage=float(np.mean(s[ti,yv])),
          oracle_containment=float(np.mean(np.all((~oracle)|s,axis=1))),
          total_shots=spent,mean_test_shots=np.nan if name=='oracle_ideal' else (
            float({'V1_adaptive':s1,'V2_rank_upper':s2a,'V2_endpoint':s2b}.get(name,0))/len(yv) if name in ('V1_adaptive','V2_rank_upper','V2_endpoint') else
            float(name.split('_')[-1]) if name.startswith('fixed_') else 64.0),
          test_accuracy=float(np.mean((pv>=.5)==yv)),
          alpha=ALPHA,beta=BETA,q_oracle=oracleq,q_v1=old['qhi'],q_v2=newq,q_fixed_1024=qfix[1024],
          v1_cal_shots=old['shots'],v2_subset_v1=bool(np.all((~v2a)|v1)),
          endpoint_fixed1024_match_fraction=endpoint_match_fraction))
    return rows


def paired_summary(df):
    gg=df.groupby('method',sort=False)
    agg=gg.agg(mean_set=('mean_set_size','mean'),sd_set=('mean_set_size','std'),
               mean_coverage=('coverage','mean'),mean_containment=('oracle_containment','mean'),
               mean_shots=('total_shots','mean')).reset_index()
    pivot=df.pivot_table(index=['dataset','seed'],columns='method',values=['total_shots','mean_set_size'])
    baseline=pivot[('total_shots','V1_adaptive')]
    fixed=pivot[('total_shots','fixed_certified_1024')]
    for i,row in agg.iterrows():
        m=row['method']
        if m!='oracle_ideal':
            agg.loc[i,'shots_saved_vs_v1_pct']=100*(1-pivot[('total_shots',m)].div(baseline)).mean()
            agg.loc[i,'shots_saved_vs_fixed1024_pct']=100*(1-pivot[('total_shots',m)].div(fixed)).mean()
            agg.loc[i,'size_delta_vs_fixed1024']=float((pivot[('mean_set_size',m)]-pivot[('mean_set_size','fixed_certified_1024')]).mean())
    return agg


def audit_math():


    from scipy.stats import binom
    for t in [8,16,32,64]:
      for p in [0.02,0.10,0.3,0.5,0.8,0.98]:
        c=np.arange(t+1)
        up=one_sided_upper(c,t,0.025)
        under=binom.pmf(c,t,p)[up<p-1e-12].sum()
        if under>0.025+1e-11:raise AssertionError((t,p,under))

    from numpy.linalg import norm
    theta=.73
    state=torch.zeros((1,16),dtype=torch.float64);state[:,0]=1
    out=ry(state,torch.tensor([theta],dtype=torch.float64),0).numpy().ravel()
    expected=np.zeros(16);expected[0]=math.cos(theta/2);expected[8]=math.sin(theta/2)
    assert norm(out-expected)<1e-12

    eta_old=BETA_CAL/(2*100*len(STAGES))
    for j in range(len(STAGES)):
        enew=BETA_CAL/100*(CAL_EARLY_FRAC/(len(STAGES)-1) if j<len(STAGES)-1 else 1-CAL_EARLY_FRAC)
        assert enew>=eta_old
    assert abs(TEST_EARLY_FRAC+ (1-TEST_EARLY_FRAC)-1.)<1e-12
    return 'PASS'


def build_figures(df,outdir):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    p=paired_summary(df)
    sub=p[p['method']!='oracle_ideal']
    order=['V1_adaptive','V2_rank_upper','V2_endpoint','fixed_certified_256',
           'fixed_certified_512','fixed_certified_1024','standard_QCP_64']
    vv=sub.set_index('method').loc[order]
    plt.figure(figsize=(9,5))
    for i,(x,y) in enumerate(zip(vv['mean_shots'],vv['mean_set'])):
        plt.scatter(x/1000,y,s=48)
        plt.annotate(order[i],(x/1000,y),xytext=(4,5),textcoords='offset points',fontsize=8)
    plt.xlabel('Mean total shots per run (thousands)')
    plt.ylabel('Mean prediction-set size')
    plt.title('Certified shot-cost versus set-size trade-off')
    plt.grid(alpha=.25);plt.tight_layout();plt.savefig(outdir/'shot_cost_vs_set_size.png',dpi=180);plt.close()
    selected=['V1_adaptive','V2_rank_upper','V2_endpoint','fixed_certified_1024','standard_QCP_64']
    vals=sub.set_index('method').loc[selected]
    plt.figure(figsize=(9,4));plt.bar(range(len(selected)),vals['mean_set'])
    plt.xticks(range(len(selected)),selected,rotation=18,ha='right')
    plt.ylabel('Mean prediction-set size');plt.title('Conformal set sizes (held-out seeds)')
    plt.tight_layout();plt.savefig(outdir/'set_size.png',dpi=180);plt.close()
    plt.figure(figsize=(9,4));plt.bar(range(len(selected)),vals['mean_coverage'])
    plt.axhline(.90,color='black',ls='--',label='Nominal 90%');plt.legend()
    plt.xticks(range(len(selected)),selected,rotation=18,ha='right');plt.ylim(.75,1.02)
    plt.ylabel('Empirical test coverage');plt.tight_layout();plt.savefig(outdir/'coverage.png',dpi=180);plt.close()


def main():
    parser=argparse.ArgumentParser(description='Quantum CP certified rank-aware V2')
    parser.add_argument('--seeds',type=int,default=10,help='Number of held-out seeds')
    parser.add_argument('--seed-start',type=int,default=10,help='Evaluation seeds start at 10 (dev 0..4)')
    parser.add_argument('--epochs',type=int,default=130)
    parser.add_argument('--output',type=str,default=str(OUT/'RESULTS'))
    args=parser.parse_args()
    t0=time.time();assert audit_math()=='PASS'
    out=Path(args.output);out.mkdir(parents=True,exist_ok=True)
    rows=[]
    for ds in ['breast','wine01','digits38']:
      for seed in range(args.seed_start,args.seed_start+args.seeds):
        item=evaluate_one(ds,seed,epochs=args.epochs)
        rows.extend(item)
        v1=item[0];v2=item[2]
        print(ds,seed,'size',round(v1['mean_set_size'],3),'->',round(v2['mean_set_size'],3),
             'shots',int(v1['total_shots']),'->',int(v2['total_shots']),flush=True)
    df=pd.DataFrame(rows);df.to_csv(out/'all_method_runs.csv',index=False)
    sm=paired_summary(df);sm.to_csv(out/'summary_by_method.csv',index=False)
    by_ds=df.groupby(['dataset','method'],as_index=False).agg(coverage=('coverage','mean'),
                set_size=('mean_set_size','mean'),shots=('total_shots','mean'),
                containment=('oracle_containment','mean'))
    by_ds.to_csv(out/'summary_by_dataset.csv',index=False)
    build_figures(df,out)
    pairs=df.pivot(index=['dataset','seed'],columns='method',values=['mean_set_size','total_shots'])
    dsize=(pairs['mean_set_size']['V2_endpoint']-pairs['mean_set_size']['fixed_certified_1024']).to_numpy()
    dsaved=1-(pairs['total_shots']['V2_endpoint']/pairs['total_shots']['fixed_certified_1024']).to_numpy()
    dsaved_v1=1-(pairs['total_shots']['V2_endpoint']/pairs['total_shots']['V1_adaptive']).to_numpy()
    dsizedom=(pairs['mean_set_size']['V2_rank_upper']-pairs['mean_set_size']['V1_adaptive']).to_numpy()
    assert np.all(dsizedom<=1e-12)

    bg=np.random.default_rng(29017);ind=bg.integers(0,len(dsize),(2000,len(dsize)))
    bootstrap=lambda v:np.quantile(v[ind].mean(axis=1),[.025,.975]).tolist()
    report={
      'date':'2026-11-13','n_runs':args.seeds*3,'n_method_evaluations':len(df),
      'seeds':list(range(args.seed_start,args.seed_start+args.seeds)),
      'alpha':ALPHA,'beta':BETA,'cal_early_frac':CAL_EARLY_FRAC,'test_early_frac':TEST_EARLY_FRAC,
      'set_delta_endpoint_minus_fixed1024_mean':float(dsize.mean()),
      'set_delta_endpoint_minus_fixed1024_bootstrap95':bootstrap(dsize),
      'shots_saved_endpoint_vs_fixed1024_mean':float(dsaved.mean()),
      'shots_saved_endpoint_vs_fixed1024_bootstrap95':bootstrap(dsaved),
      'shots_saved_endpoint_vs_v1_mean':float(dsaved_v1.mean()),
      'shots_saved_endpoint_vs_v1_bootstrap95':bootstrap(dsaved_v1),
      'endpoint_set_no_larger_than_fixed1024_runs':int(np.sum(dsize<=1e-12)),
      'rank_upper_dominates_v1_all_runs':bool(np.all(dsizedom<=1e-12)),
      'endpoint_fixed1024_exact_matches_all_samples':bool(np.all(df[df.method=='V2_endpoint']['endpoint_fixed1024_match_fraction']>=1-1e-12)),
      'audit':'PASS','seconds':float(time.time()-t0),
      'data_notice':'Shot streams pre-generated to pair methods. Counts are hypothetical measured prefixes, not CPU draw counts.',
      'coverage_notice':'Marginal per-query coverage under exchangeability; no hardware-noise or time-varying guarantees.'}
    (out/'metrics.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
    print('\nHELD-OUT RESULTS\n',sm[['method','mean_set','mean_coverage','mean_shots','shots_saved_vs_fixed1024_pct']].to_string(index=False))
    print(json.dumps({k:v for k,v in report.items() if k not in ['seeds','data_notice','coverage_notice']},indent=2))

if __name__=='__main__':main()


In [ ]:
%%writefile v3_reference.py
import os, math, sys, json, time, argparse
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import binom, norm
import qcp_rank_v2 as base
from pilot_v3 import evidence_readout, audit_e_process

STAGES=base.STAGES
OUT=Path(__file__).resolve().parent/'RESULTS'

def calibrated_rho(n):


    return max(.2,min(.7,32./n))

def test_with_spending(qhi,streams,rho):
    n=len(streams);lo=np.zeros(n);hi=np.ones(n)
    active=np.ones(n,dtype=bool);stage=np.zeros(n,dtype=int)
    weights=np.full(len(STAGES),rho/(len(STAGES)-1));weights[-1]=1-rho
    for j,t in enumerate(STAGES):
        ix=np.flatnonzero(active)
        if not len(ix):break
        ll,uu=base.cp_interval(streams[ix,t-1],t,base.BETA_TEST*weights[j])
        lo[ix],hi[ix]=base.update_nested(lo[ix],hi[ix],ll,uu)
        stage[ix]=j+1
        scorelo=np.column_stack([lo[ix],1-hi[ix]])
        scorehi=np.column_stack([hi[ix],1-lo[ix]])
        active[ix[np.all((scorehi<=qhi)|(scorelo>qhi),axis=1)]]=False
    sets=np.column_stack([lo<=qhi,1-hi<=qhi])
    return sets,int(np.sum(np.asarray(STAGES)[stage-1])),stage

def ranktail_gamma(n,m,eta=base.BETA_CAL):
    if m<0 or m>=n:raise ValueError((n,m))
    return brentq(lambda g:binom.sf(m,n,g)-eta,1e-12,.5,xtol=1e-14)

def ranktail_quantile(y,calstream,t=256):
    n=len(y);k=math.ceil((n+1)*(1-base.ALPHA+base.BETA))
    if k>n:return 1.,0,0.
    m=max(0,min(n-k,max(1,int(.025*n))))
    gamma=ranktail_gamma(n,m)
    raw=calstream[:,t-1]
    succ=np.where(y==0,raw,t-raw)
    uppers=base.one_sided_upper(succ,t,gamma)
    return base.order_threshold(uppers,k+m),m,gamma

METHODS=[
 'V1_adaptive','V2_endpoint','V3_size_adaptive','V3_rho_020','V3_eprocess_4tests',
 'V3_ranktail_cal_256','fixed_certified_256','fixed_certified_512',
 'fixed_certified_1024','standard_QCP_64','oracle_ideal']

def one_run(ds,seed,epochs=130):
    (zt,yt),(zc,yc),(zv,yv)=base.prepare(ds,seed)
    th=base.train((zt,yt),seed,epochs)
    pc,pv=base.predict(zc,th),base.predict(zv,th)
    rng=np.random.default_rng(4321+seed*1009+len(ds)*113)
    cal=base.sample_prefixes(pc,rng);test=base.sample_prefixes(pv,rng)
    old=base.old_calibration(yc,cal);qhi,_=base.quantile_focused_upper(yc,cal,old['stage'])
    qtrue=base.order_threshold(np.where(yc==0,pc,1-pc),old['k'])
    ideal=np.column_stack([pv<=qtrue,1-pv<=qtrue])
    v1,c1,_=base.test_v1(old['qlo'],old['qhi'],test)
    v2,c2,_=base.test_v2(qhi,test,True)
    rho=calibrated_rho(len(yc));v3,c3,s3=test_with_spending(qhi,test,rho)
    v3_fix,ccfix,sfix=test_with_spending(qhi,test,.20)
    ve,ce,se,_=evidence_readout(qhi,test)
    qtail,m,gamma=ranktail_quantile(yc,cal,256)
    vt,ct,_=base.test_v2(qtail,test,True)


    fixed={t:base.fixed_certificate(yc,cal,test,t) for t in (256,512,1024)}
    qcp,cqcp,_=base.ordinary_qcp(yc,cal,test,64)
    methods={
       'V1_adaptive':(v1,old['shots']+c1),
       'V2_endpoint':(v2,old['shots']+c2),
       'V3_size_adaptive':(v3,old['shots']+c3),
       'V3_rho_020':(v3_fix,old['shots']+ccfix),
       'V3_eprocess_4tests':(ve,old['shots']+ce),
       'V3_ranktail_cal_256':(vt,len(yc)*256+ct),
       'fixed_certified_256':(fixed[256][0],fixed[256][1]),
       'fixed_certified_512':(fixed[512][0],fixed[512][1]),
       'fixed_certified_1024':(fixed[1024][0],fixed[1024][1]),
       'standard_QCP_64':(qcp,cqcp),
       'oracle_ideal':(ideal,np.nan),
    }
    rows=[];details=[]
    for method in METHODS:
        sets,shots=methods[method]
        rows.append(dict(dataset=ds,seed=seed,method=method,n_train=len(yt),n_cal=len(yc),n_test=len(yv),
            rho=rho,qhi=qhi,qtrue=qtrue,ranktail_m=m,ranktail_gamma=gamma,
            total_shots=shots,mean_set_size=float(sets.sum(axis=1).mean()),
            coverage=float(sets[np.arange(len(yv)),yv].mean()),
            oracle_containment=float(np.all((~ideal)|sets,axis=1).mean()),
            set_equal_v2=float(np.all(sets==v2,axis=1).mean()),
            set_size_delta_v2=float((sets.sum(axis=1)-v2.sum(axis=1)).mean()),
            v1_cal_shots=int(old['shots']),
            classifier_accuracy=float(((pv>=.5)==yv).mean()),
            certification_target_beta=base.BETA))
    for i,p in enumerate(pv):
        details.append(dict(dataset=ds,seed=seed,index=i,prob=float(p),
          qhi=float(qhi),margin=float(min(abs(p-qhi),abs(p-(1-qhi)))),
          V3_test_shots=int(np.asarray(STAGES)[s3[i]-1]),
          eprocess_test_shots=int(se[i]),
          V3_set_size=int(v3[i].sum()),V2_set_size=int(v2[i].sum()),
          truth=int(yv[i]),V3_covered=bool(v3[i,yv[i]])))
    return rows,details

def kl_bern(p,q):
    p=np.asarray(p,dtype=float);q=np.asarray(q,dtype=float)
    return p*np.log(p/q)+(1-p)*np.log((1-p)/(1-q))

def kl_decision_lower_bound(c,g,err):


    if not (0<err<.5 and 0<g<min(c,1-c)):return float('nan')
    p0=c-g;p1=c+g
    d=(1-err)*np.log((1-err)/err)+err*np.log(err/(1-err))
    return float(d/kl_bern(p1,p0))

def audit_quantile():

    for n,k in [(142,131),(89,82),(32,31)]:
        m=min(n-k,max(1,int(.025*n)));g=ranktail_gamma(n,m)
        assert abs(binom.sf(m,n,g)-base.BETA_CAL)<1e-9
        rng=np.random.default_rng(920+n)
        for _ in range(250):
            actual=rng.uniform(size=n)
            u=np.minimum(1,actual+rng.uniform(0,.5,n))
            bad=rng.choice(n,size=m,replace=False)
            u[bad]=np.maximum(0,actual[bad]-.15)
            assert base.order_threshold(actual,k) <= base.order_threshold(u,k+m)+1e-12
    assert audit_e_process()=='PASS'

    for g in [.025,.05,.1,.2]:
        assert kl_decision_lower_bound(.5,g,.01)>0
    return 'PASS'

def plot_results(df,debug,out):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    gp=df.groupby('method').agg(shots=('total_shots','mean'),size=('mean_set_size','mean'),coverage=('coverage','mean'))
    order=[m for m in METHODS if m!='oracle_ideal']
    fig,ax=plt.subplots(figsize=(9,5))
    for m in order:
        r=gp.loc[m];ax.scatter(r.shots/1000,r['size'],s=48)
        ax.annotate(m,(r.shots/1000,r['size']),xytext=(4,4),textcoords='offset points',fontsize=8)
    ax.set_xlabel('Mean total shots per dataset-seed run (thousands)');ax.set_ylabel('Mean prediction-set size')
    ax.set_title('Coverage certificate: shot cost vs set size');ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(out/'shot_vs_size.png',dpi=180);plt.close(fig)
    fig,ax=plt.subplots(figsize=(9,4))
    keep=['V2_endpoint','V3_size_adaptive','V3_eprocess_4tests','V3_ranktail_cal_256','fixed_certified_1024']
    axs=np.arange(len(keep));ax.bar(axs,[gp.loc[m,'shots']/1000 for m in keep]);ax.set_xticks(axs)
    ax.set_xticklabels(keep,rotation=20,ha='right',fontsize=9);ax.set_ylabel('Mean total shots (thousands)')
    fig.tight_layout();fig.savefig(out/'shots_bar.png',dpi=180);plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,4))
    x=np.maximum(debug['margin'].to_numpy(),1e-4)
    y=debug['V3_test_shots'].to_numpy()
    ax.scatter(x,y,alpha=.35,s=10)
    ax.set_xscale('log');ax.set_xlabel('Gap to nearest conformal decision boundary')
    ax.set_ylabel('V3 test shots (predeclared stages)');ax.set_title('Boundary difficulty and sample cost')
    ax.grid(alpha=.2);fig.tight_layout();fig.savefig(out/'margin_vs_shots.png',dpi=180);plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,4));c=.5;err=.01
    gs=np.geomspace(.005,.2,80)
    ax.plot(gs,[kl_decision_lower_bound(c,g,err) for g in gs])
    ax.set_xscale('log');ax.set_yscale('log')
    ax.set_xlabel('Separation g (Bernoulli probabilities 0.5-g vs 0.5+g)')
    ax.set_ylabel('Information-theoretic lower bound on E[T]')
    ax.set_title('Reference: two-sided fixed-confidence threshold decision')
    ax.grid(alpha=.2);fig.tight_layout();fig.savefig(out/'kl_lower_bound.png',dpi=180);plt.close(fig)

def main():
    p=argparse.ArgumentParser()
    p.add_argument('--seeds',type=int,default=10)
    p.add_argument('--seed-start',type=int,default=30)
    p.add_argument('--epochs',type=int,default=130)
    p.add_argument('--output',default=str(OUT))
    args=p.parse_args();out=Path(args.output);out.mkdir(exist_ok=True,parents=True)
    assert base.audit_math()=='PASS';assert audit_quantile()=='PASS'
    time0=time.time();rr=[];dd=[]
    for ds in ['breast','wine01','digits38']:
        for seed in range(args.seed_start,args.seed_start+args.seeds):
            r,d=one_run(ds,seed,args.epochs);rr.extend(r);dd.extend(d)
            v2=r[METHODS.index('V2_endpoint')];v3=r[METHODS.index('V3_size_adaptive')]
            print(f'{ds:8s} seed {seed}: V2 {v2["total_shots"]:6.0f} -> V3 {v3["total_shots"]:6.0f} '
                  f'set {v2["mean_set_size"]:.3f} -> {v3["mean_set_size"]:.3f}',flush=True)
    df=pd.DataFrame(rr);debug=pd.DataFrame(dd)
    df.to_csv(out/'all_methods_runs.csv',index=False)
    debug.to_csv(out/'test_sample_details.csv',index=False)
    sm=df.groupby('method',as_index=False).agg(mean_shots=('total_shots','mean'),sd_shots=('total_shots','std'),
        mean_set=('mean_set_size','mean'),mean_coverage=('coverage','mean'),oracle_containment=('oracle_containment','mean'))
    sm.to_csv(out/'summary.csv',index=False)
    byds=df.groupby(['dataset','method'],as_index=False).agg(mean_shots=('total_shots','mean'),mean_set=('mean_set_size','mean'),mean_coverage=('coverage','mean'))
    byds.to_csv(out/'by_dataset.csv',index=False)
    pivot=df.pivot(index=['dataset','seed'],columns='method',values=['total_shots','mean_set_size'])
    v2s=pivot['total_shots']['V2_endpoint']
    v3s=pivot['total_shots']['V3_size_adaptive']
    size2=pivot['mean_set_size']['V2_endpoint'];size3=pivot['mean_set_size']['V3_size_adaptive']
    save=(1-v3s/v2s).to_numpy();dsize=(size3-size2).to_numpy()
    rng=np.random.default_rng(121);indices=rng.integers(0,len(save),size=(10000,len(save)))
    boot=lambda arr:np.quantile(arr[indices].mean(axis=1),[.025,.975]).tolist()
    metric={'date':'2026-11-13','n_datasets':3,'seeds':args.seeds,'start_seed':args.seed_start,
      'n_fit_runs':args.seeds*3,'n_method_evaluations':len(df),'n_test_points':len(debug),
      'audit': 'PASS', 'alpha':base.ALPHA,'beta':base.BETA,
      'mean_saving_vs_V2_fraction':float(save.mean()),'bootstrap_saving_fraction_95':boot(save),
      'mean_set_delta_vs_V2':float(dsize.mean()),'bootstrap_set_delta_95':boot(dsize),
      'same_or_smaller_set_runs':int(np.sum(dsize<=1e-12)),
      'all_runs':len(dsize), 'mean_containment_V3':float(df[df.method=='V3_size_adaptive'].oracle_containment.mean()),
      'runtime_seconds':float(time.time()-time0),
      'caveat':'Bootstrap is descriptive over dataset-seed blocks; only three related source datasets.',
      'shot_count':'Hypothetical physical executions of measured prefixes; full counterfactual streams pre-generated for pairing.',
      'coverage_notice':'Per-query marginal coverage; standard QCP64 is a different finite-shot guarantee and not ideal-oracle containment.'}
    (out/'metrics.json').write_text(json.dumps(metric,indent=2))
    plot_results(df,debug,out)
    print('\nFINAL RESULTS\n',sm.to_string(index=False))
    print('\nMETRICS\n',json.dumps(metric,indent=2))

if __name__=='__main__':main()


In [ ]:
%%writefile pilot_v3.py
import os, sys, math, time, argparse, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import logsumexp

try: import qcp_rank_v2 as base
except ModuleNotFoundError:
    sys.path.insert(0,str(Path(__file__).resolve().parent))
    import qcp_rank_v2 as base

OUT=Path(__file__).resolve().parent/'RESULTS'
STAGES=np.array(base.STAGES)
BETA_TEST=base.BETA_TEST

FRACS=np.array([.025,.05,.10,.20,.35,.50,.70,.90])
WEIGHTS=np.array([.07,.10,.14,.20,.19,.14,.10,.06]); assert abs(WEIGHTS.sum()-1)<1e-10


def log_e_mix(count,t,c,direction):


    count=np.asarray(count,dtype=float);c=np.broadcast_to(np.asarray(c,dtype=float),count.shape)
    ans=np.full(count.shape,-np.inf)
    valid=(c>0)&(c<1)
    if not valid.any(): return ans
    cv=c[valid];nv=count[valid]
    if direction=='up': alt=cv[:,None]+(1-cv[:,None])*FRACS[None,:]
    elif direction=='down': alt=cv[:,None]*(1-FRACS[None,:])
    else:raise ValueError(direction)
    log_lrs=nv[:,None]*np.log(alt/cv[:,None])+(t-nv[:,None])*np.log((1-alt)/(1-cv[:,None]))
    ans[valid]=logsumexp(log_lrs+np.log(WEIGHTS),axis=1)


    return ans


def evidence_readout(qhi,stream,cap=1024,split_budget=4,stages=STAGES):


    n=len(stream);alive=np.ones(n,dtype=bool);shots=np.zeros(n,dtype=int)
    result=np.ones((n,2),dtype=bool);status=np.zeros((n,2),dtype=np.int8)
    threshold=np.log(split_budget/BETA_TEST)
    for t in stages:
        if t>cap:break
        ix=np.flatnonzero(alive)
        if not len(ix):break
        c=stream[ix,t-1];b0=np.full(len(ix),float(qhi));b1=1-b0
        e_0_ex=log_e_mix(c,t,b0,'up')>=threshold
        e_0_in=log_e_mix(c,t,b0,'down')>=threshold
        e_1_ex=log_e_mix(c,t,b1,'down')>=threshold
        e_1_in=log_e_mix(c,t,b1,'up')>=threshold
        st0=np.where(e_0_ex,-1,np.where(e_0_in,1,0))
        st1=np.where(e_1_ex,-1,np.where(e_1_in,1,0))
        status[ix,0]=st0;status[ix,1]=st1
        result[ix,0]=st0!=-1;result[ix,1]=st1!=-1
        shots[ix]=t


        resolved=(np.all(status[ix]!=0,axis=1) | ((qhi>=.5) & np.any(status[ix]==-1,axis=1)))
        alive[ix[resolved]]=False
    return result,int(shots.sum()),shots,status


def test_evidence_exclusion(qhi,stream,cap=1024,stages=STAGES):


    n=len(stream);active=np.ones(n,dtype=bool);spent=np.zeros(n,dtype=int)
    sets=np.ones((n,2),dtype=bool);thr=math.log(2/BETA_TEST)
    for t in stages:
        if t>cap:break
        ix=np.flatnonzero(active)
        if len(ix)==0:break
        counts=stream[ix,t-1]
        mask0=log_e_mix(counts,t,np.full(len(ix),qhi),'up')>=thr
        mask1=log_e_mix(counts,t,np.full(len(ix),1-qhi),'down')>=thr
        sets[ix,0]=~mask0;sets[ix,1]=~mask1
        spent[ix]=t
        if qhi>=.5: active[ix[mask0|mask1]]=False
        else: active[ix[mask0&mask1]]=False
    return sets,int(spent.sum()),spent


def evaluate_one(ds,seed,epochs=130):
    (zt,yt),(zc,yc),(zv,yv)=base.prepare(ds,seed)
    theta=base.train((zt,yt),seed,epochs)
    pc,pv=base.predict(zc,theta),base.predict(zv,theta)
    rng=np.random.default_rng(4321+seed*1009+len(ds)*113)
    cal=base.sample_prefixes(pc,rng);test=base.sample_prefixes(pv,rng)
    old=base.old_calibration(yc,cal)
    qhi,_=base.quantile_focused_upper(yc,cal,old['stage'])
    v1,s1,_=base.test_v1(old['qlo'],old['qhi'],test)
    v2,s2,_=base.test_v2(qhi,test,True)
    e4,s4,sper,_=evidence_readout(qhi,test)
    e2,st2,sp2=test_evidence_exclusion(qhi,test)
    fixed,sfix,_=base.fixed_certificate(yc,cal,test,1024)
    qcp,sqcp,_=base.ordinary_qcp(yc,cal,test,64)
    k=old['k']; qtrue=base.order_threshold(np.where(yc==0,pc,1-pc),k)
    ideal=np.column_stack([pv<=qtrue,1-pv<=qtrue])
    methods={
      'V1_adaptive':(v1,old['shots']+s1),
      'V2_endpoint':(v2,old['shots']+s2),
      'V3_eprocess_4tests':(e4,old['shots']+s4),
      'V3_eprocess_2tests':(e2,old['shots']+st2),
      'fixed_certified_1024':(fixed,sfix),
      'ordinary_QCP_64':(qcp,sqcp)
    }
    rows=[]
    for name,(sets,nshots) in methods.items():
        rows.append({'dataset':ds,'seed':seed,'method':name,'n_cal':len(yc),'n_test':len(yv),
          'q_up':qhi,'q_ideal':qtrue,
          'shots_total':nshots,'set_size':float(sets.sum(axis=1).mean()),
          'coverage':float(sets[np.arange(len(yv)),yv].mean()),
          'oracle_containment':float(np.all((~ideal)|sets,axis=1).mean()),
          'test_shots':int(nshots-old['shots']) if name not in ('fixed_certified_1024','ordinary_QCP_64') else np.nan,
          'set_delta_vs_v2':float((sets.sum(axis=1)-v2.sum(axis=1)).mean()),
          'fraction_exact_same_as_v2':float(np.all(sets==v2,axis=1).mean()),
          'cal_shots':old['shots'],'accuracy':float(((pv>=.5)==yv).mean())})

    debug=[]
    for i,p in enumerate(pv):
        debug.append({'dataset':ds,'seed':seed,'test_i':i,'p':float(p),'qhi':qhi,
           'gap_to_exclusion_boundary':float(min(abs(p-qhi),abs(p-(1-qhi)))),
           'e4_shots':int(sper[i]),'e2_shots':int(sp2[i]),
           'v2_set_size':int(v2[i].sum()),'e4_set_size':int(e4[i].sum()),
           'v2_includes_true':bool(v2[i,yv[i]]),'e4_includes_true':bool(e4[i,yv[i]])})
    return rows,debug


def audit_e_process():

    from scipy.stats import binom
    for direction in ['up','down']:
        c=.35
        ps=np.linspace(0,c,8) if direction=='up' else np.linspace(c,1,8)
        for p in ps:


            for t in [8,32,64]:
                ks=np.arange(t+1)
                E=log_e_mix(ks,t,np.full(len(ks),c),direction)
                prob=binom.pmf(ks,t,p)[E>=math.log(1/.05)].sum()
                assert prob<=.05+1e-10,(direction,t,p,prob)
    return 'PASS'


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--seed-start',type=int,default=20)
    parser.add_argument('--seeds',type=int,default=10)
    parser.add_argument('--epochs',type=int,default=130)
    parser.add_argument('--output',default=str(OUT))
    a=parser.parse_args()
    assert audit_e_process()=='PASS'; t0=time.time()
    out=Path(a.output);out.mkdir(parents=True,exist_ok=True)
    rows=[];debug=[]
    for ds in ['breast','wine01','digits38']:
        for seed in range(a.seed_start,a.seed_start+a.seeds):
            r,d=evaluate_one(ds,seed,a.epochs);rows+=r;debug+=d
            print(ds,seed,'V2',int(r[1]['shots_total']),'e4',int(r[2]['shots_total']),
                  'e2',int(r[3]['shots_total']), 'size',round(r[1]['set_size'],3),round(r[2]['set_size'],3),flush=True)
    df=pd.DataFrame(rows);dbg=pd.DataFrame(debug)
    df.to_csv(out/'all_runs.csv',index=False);dbg.to_csv(out/'per_test_debug.csv',index=False)
    agg=df.groupby('method',as_index=False).agg(shots=('shots_total','mean'),size=('set_size','mean'),coverage=('coverage','mean'),containment=('oracle_containment','mean'))
    agg.to_csv(out/'summary.csv',index=False)
    piv=df.pivot(index=['dataset','seed'],columns='method',values=['shots_total','set_size'])
    res={'seed_start':a.seed_start,'seeds':a.seeds,'n_runs':len(df),'n_test_samples':len(dbg),
      'audit_eprocess':audit_e_process(),
      'savings_e4_vs_v2_pct':float((100*(1-piv['shots_total']['V3_eprocess_4tests']/piv['shots_total']['V2_endpoint'])).mean()),
      'savings_e2_vs_v2_pct':float((100*(1-piv['shots_total']['V3_eprocess_2tests']/piv['shots_total']['V2_endpoint'])).mean()),
      'size_delta_e4_vs_v2':float((piv['set_size']['V3_eprocess_4tests']-piv['set_size']['V2_endpoint']).mean()),
      'size_delta_e2_vs_v2':float((piv['set_size']['V3_eprocess_2tests']-piv['set_size']['V2_endpoint']).mean()),
      'elapsed_seconds':time.time()-t0}
    (out/'metrics.json').write_text(json.dumps(res,indent=2))
    print(agg.to_string(index=False));print(json.dumps(res,indent=2))

if __name__=='__main__':main()


In [ ]:
%%writefile experiment_width_matched.py
import argparse, json, math, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import qcp_rank_v2 as b
from v3_reference import calibrated_rho, test_with_spending

ROOT=Path(__file__).resolve().parent
SPLIT_POLICY='original_stratified'
ST=np.asarray(b.STAGES); NST=len(ST)
DATASETS=['breast','wine01','digits38']

CAL_THRESHOLDS=[0.10,0.14,0.18,0.22,0.27,0.33,0.42]
TEST_THRESHOLDS=[0.08,0.12,0.16,0.22,0.30]
WIDTH_PRIORITY_POWERS=[0.0,0.5,1.0]
TOL_SET=0.005                                                                           


def score_cp_width(count,t,err):
    lo,hi=b.cp_interval(count,t,err)
    return hi-lo


def width_cal_threshold(y, stream, threshold):


    n=len(y);stage=np.zeros(n,dtype=np.int8);active=np.ones(n,dtype=bool)
    e=b.BETA_CAL/(n*NST)
    for j,t in enumerate(ST):
        ix=np.flatnonzero(active)
        if not len(ix):break
        c=stream[ix,t-1]
        w=score_cp_width(c,t,e)
        stage[ix]=j+1
        active[ix[w<=threshold]]=False
    qhi,_=b.quantile_focused_upper(y,stream,stage)
    shots=int(np.sum(ST[stage-1]))
    return qhi,shots,stage


def width_cal_budgeted(y,stream, budget, priority_power=0.):


    n=len(y); stage=np.ones(n,dtype=np.int8)
    cost=int(ST[0]*n);budget=max(cost,min(int(budget),int(ST[-1]*n)))
    e=b.BETA_CAL/(n*NST)
    width=score_cp_width(stream[:,ST[0]-1],int(ST[0]),e)
    while True:
        can=np.flatnonzero(stage<NST)
        if not len(can):break
        extra=ST[stage[can]]-ST[stage[can]-1]
        can=can[extra<=budget-cost]
        if not len(can):break

        next_cost=ST[stage[can]]-ST[stage[can]-1]
        priority=width[can]/np.power(next_cost,priority_power)
        idx=can[np.argmax(priority)]
        new_j=stage[idx]
        extra=int(ST[new_j]-ST[new_j-1])
        stage[idx]+=1;cost+=extra
        t=int(ST[stage[idx]-1]);c=int(stream[idx,t-1])
        width[idx]=score_cp_width(np.array([c]),t,e)[0]
    qhi,_=b.quantile_focused_upper(y,stream,stage)
    return qhi,cost,stage


def width_only_test(qhi,stream,threshold,rho):


    n=len(stream);lo=np.zeros(n);hi=np.ones(n)
    stage=np.zeros(n,dtype=np.int8);active=np.ones(n,dtype=bool)
    weights=np.full(NST,rho/(NST-1));weights[-1]=1-rho
    for j,t in enumerate(ST):
        ix=np.flatnonzero(active)
        if not len(ix):break
        ll,uu=b.cp_interval(stream[ix,t-1],t,b.BETA_TEST*weights[j])
        lo[ix],hi[ix]=b.update_nested(lo[ix],hi[ix],ll,uu)
        stage[ix]=j+1
        active[ix[(hi[ix]-lo[ix])<=threshold]]=False
    sets=np.column_stack([lo<=qhi,1-hi<=qhi])
    return sets,int(np.sum(ST[stage-1])),stage


def prepare_random_holdout(ds,seed):


    from sklearn.datasets import load_breast_cancer,load_wine,load_digits
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    if ds=='breast':
        dat=load_breast_cancer();X=dat.data;y=dat.target
    elif ds=='wine01':
        dat=load_wine();m=dat.target<2;X=dat.data[m];y=dat.target[m]
    elif ds=='digits38':
        dat=load_digits();m=np.isin(dat.target,[3,8]);X=dat.data[m];y=(dat.target[m]==8).astype(int)
    else:raise ValueError(ds)
    indices=np.arange(len(y))
    a,rest=train_test_split(indices,test_size=.5,stratify=y,random_state=seed)
    cal,test=train_test_split(rest,test_size=.5,random_state=seed+101,stratify=None)
    scale=StandardScaler().fit(X[a]);xx=scale.transform(X)
    pca=PCA(n_components=4,random_state=seed).fit(xx[a]);zz=pca.transform(xx)
    scale2=StandardScaler().fit(zz[a]);zz=np.pi*.78*np.tanh(scale2.transform(zz)/1.8)
    return (zz[a],y[a]),(zz[cal],y[cal]),(zz[test],y[test])


def init_run(ds,seed,epochs):
    (zt,yt),(zc,yc),(zv,yv)=(prepare_random_holdout(ds,seed) if SPLIT_POLICY=='random_holdout' else b.prepare(ds,seed))
    theta=b.train((zt,yt),seed,epochs)
    pc,pv=b.predict(zc,theta),b.predict(zv,theta)
    rng=np.random.default_rng(4321+seed*1009+len(ds)*113)
    cal=b.sample_prefixes(pc,rng);test=b.sample_prefixes(pv,rng)
    old=b.old_calibration(yc,cal)
    qrank,_=b.quantile_focused_upper(yc,cal,old['stage'])
    rho=calibrated_rho(len(yc))
    rankset,ranktestshots,rankstage=test_with_spending(qrank,test,rho)
    qideal=b.order_threshold(np.where(yc==0,pc,1-pc),old['k'])
    ideal=np.column_stack([pv<=qideal,1-pv<=qideal])
    return dict(ds=ds,seed=seed,yc=yc,yv=yv,cal=cal,test=test,pv=pv,
      qideal=qideal,ideal=ideal,old=old,qrank=qrank,rho=rho,
      rankset=rankset,ranktestshots=ranktestshots,rankstage=rankstage)


def row(run,method,sets,calshots,testshots,qhi,stage_cal=None,stage_test=None,policy=''):
    y=run['yv'];idx=np.arange(len(y));ideal=run['ideal']
    return dict(dataset=run['ds'],seed=run['seed'],method=method,policy=policy,
      n_cal=len(run['yc']),n_test=len(y),cal_shots=int(calshots),test_shots=int(testshots),
      total_shots=int(calshots+testshots),mean_set_size=float(sets.sum(1).mean()),
      coverage=float(sets[idx,y].mean()),
      ideal_set_containment=float(np.all((~ideal)|sets,axis=1).mean()),
      exact_set_match_rank=float(np.all(sets==run['rankset'],axis=1).mean()),
      subset_rank=float(np.all((~sets)|run['rankset'],axis=1).mean()),
      qhi=float(qhi),rank_qhi=float(run['qrank']),
      mean_cal_shots=float(np.mean(ST[stage_cal-1])) if stage_cal is not None else np.nan,
      mean_test_shots=float(np.mean(ST[stage_test-1])) if stage_test is not None else np.nan)


def dev_candidates(run):

    records=[]
    cal_cache={t:width_cal_threshold(run['yc'],run['cal'],t) for t in CAL_THRESHOLDS}
    for ct,(q,calshots,calst) in cal_cache.items():
        for tt in TEST_THRESHOLDS:
            sets,testshots,testst=width_only_test(q,run['test'],tt,run['rho'])
            key=f'width_cal{ct:.2f}_test{tt:.2f}'
            records.append(row(run,key,sets,calshots,testshots,q,calst,testst,policy=f'{ct:.2f}|{tt:.2f}'))
    return records


def select_on_dev(df,rank_df):
    rsz=float(rank_df['mean_set_size'].mean())
    target_coverage=.90
    v=df.groupby('method',as_index=False).agg(shots=('total_shots','mean'),
     size=('mean_set_size','mean'),coverage=('coverage','mean'),
     containment=('ideal_set_containment','mean'))
    v['size_gap_vs_rank']=v['size']-rsz

    ok=v[(v['size']<=rsz+TOL_SET)&(v['coverage']>=target_coverage)&(v['containment']>=.99)]
    if len(ok):
        selected=ok.sort_values(['shots','size','method']).iloc[0]
        status='feasible'
    else:

        v['violation']=np.maximum(0,v['size']-rsz-TOL_SET)
        selected=v.sort_values(['violation','shots','method']).iloc[0]
        status='NOT_MATCHED_ON_DEV'
    return selected['method'],status,v


def quality_stats(d,rank='rank_V3',vs='width_selected'):
    p=d.pivot(index=['dataset','seed'],columns='method',values=['total_shots','mean_set_size','coverage','ideal_set_containment'])
    out={}
    for method in [x for x in d.method.unique() if x!=rank]:
        ds=(p['mean_set_size'][method]-p['mean_set_size'][rank]).to_numpy()
        shot=(p['total_shots'][method]/p['total_shots'][rank]).to_numpy()
        cov=(p['coverage'][method]-p['coverage'][rank]).to_numpy()
        out[method]={
          'ratio_shots_mean':float(np.mean(shot)),
          'mean_paired_shot_saving_pct':float(100*np.mean(1-shot)),
          'mean_paired_set_delta':float(np.mean(ds)),
          'mean_coverage_delta':float(np.mean(cov)),
          'mean_set_nonlarger_fraction':float(np.mean(ds<=1e-12)),
          'mean_set_within_tolerance_fraction':float(np.mean(np.abs(ds)<=TOL_SET+1e-12)),
          'certification_scope':('finite-shot marginal coverage, not ideal-oracle containment' if method=='standard_QCP64' else 'ideal-score containment with beta failure budget'),
        }

        rng=np.random.default_rng(70917)
        groups=[np.where(p.index.get_level_values(0)==dataset)[0] for dataset in DATASETS]
        boot_sh=[];boot_sz=[]
        for _ in range(1600):
            dsi=rng.integers(0,len(groups),size=len(groups))
            ix=np.concatenate([rng.choice(groups[z],size=len(groups[z]),replace=True) for z in dsi])
            boot_sh.append(np.mean(100*(1-shot[ix])));boot_sz.append(np.mean(ds[ix]))
        out[method]['cluster_bootstrap95_shot_saving_pct']=list(np.quantile(boot_sh,[.025,.975]))
        out[method]['cluster_bootstrap95_set_delta']=list(np.quantile(boot_sz,[.025,.975]))
    return out


def plots(hold,dev_grid,output):
    agg=hold.groupby('method').agg(shots=('total_shots','mean'),size=('mean_set_size','mean'),coverage=('coverage','mean'))
    display=['standard_QCP64','rank_V3','V2_endpoint','width_budget_matched','width_test_only','width_selected']
    labels={'rank_V3':'Rank-aware V3','width_selected':'Width-only (quality-matched)',
      'width_budget_matched':'Width-first (budget-matched)',
      'width_test_only':'Width-only test','V2_endpoint':'V2 Endpoint','standard_QCP64':'Standard QCP-64'}
    dy={'standard_QCP64':7,'rank_V3':-22,'V2_endpoint':10,
        'width_budget_matched':7,'width_test_only':-18,'width_selected':8}
    fig,ax=plt.subplots(figsize=(9.5,5.3))
    for name in display:
        t=agg.loc[name];ax.scatter(t.shots/1000,t['size'],s=70)
        ax.annotate(labels[name],(t.shots/1000,t['size']),xytext=(6,dy[name]),textcoords='offset points',fontsize=8.5)
    ax.set(xlabel='Total shots per dataset-seed run (thousands)',ylabel='Mean prediction-set size',
           title='Held-out shot cost and prediction-set quality')
    ax.margins(x=.18,y=.16);ax.grid(alpha=.2);fig.tight_layout();fig.savefig(output/'heldout_shots_vs_size.png',dpi=180);plt.close(fig)
    fig,ax=plt.subplots(figsize=(9,5.1))
    a=dev_grid[['shots','size']]
    ax.scatter(a.shots/1000,a['size'],s=20,alpha=.30,label='Dev grid (width-only)')
    for name in ['rank_V3','width_selected','width_budget_matched']:
        v=agg.loc[name]
        ax.scatter([v.shots/1000],[v['size']],s=90,label=labels[name])
    ax.set(xlabel='Total shots per run (thousands)',ylabel='Mean prediction-set size',
           title='Development frontier and held-out outcomes')
    ax.grid(alpha=.2);ax.legend(fontsize=8);fig.tight_layout();fig.savefig(output/'quality_frontier.png',dpi=180);plt.close(fig)
    fig,ax=plt.subplots(figsize=(9,4));order=['rank_V3','V2_endpoint','width_budget_matched','width_test_only','width_selected']
    ax.bar(range(len(order)),agg.loc[order,'shots']/1000)
    ax.set_xticks(range(len(order)));ax.set_xticklabels([labels[m] for m in order],rotation=16,ha='right',fontsize=9)
    ax.set_ylabel('Mean total shots (thousands)');fig.tight_layout();fig.savefig(output/'heldout_shots.png',dpi=180);plt.close(fig)


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--dev-seeds',type=int,default=5)
    parser.add_argument('--eval-seeds',type=int,default=10)
    parser.add_argument('--eval-start',type=int,default=40)
    parser.add_argument('--epochs',type=int,default=130)
    parser.add_argument('--split-policy',choices=['original_stratified','random_holdout'],default='original_stratified')
    parser.add_argument('--output',default=str(ROOT/'RESULTS'))
    args=parser.parse_args();out=Path(args.output);out.mkdir(exist_ok=True,parents=True)
    global SPLIT_POLICY
    SPLIT_POLICY=args.split_policy
    assert b.audit_math()=='PASS'
    t0=time.time()
    devrows=[];devrank=[];devbudget=[]
    for ds in DATASETS:
        for seed in range(args.dev_seeds):
            run=init_run(ds,seed,args.epochs)
            devrows+=dev_candidates(run)
            devrank.append(row(run,'rank_V3',run['rankset'],run['old']['shots'],run['ranktestshots'],run['qrank'],run['old']['stage'],run['rankstage']))
            for power in WIDTH_PRIORITY_POWERS:
                q,cs,st=width_cal_budgeted(run['yc'],run['cal'],run['old']['shots'],power)
                s,ts,stt=test_with_spending(q,run['test'],run['rho'])
                devbudget.append(row(run,f'width_priority_{power:.1f}',s,cs,ts,q,st,stt,policy=f'power={power}'))
            print('DEV',ds,seed,flush=True)
    dev=pd.DataFrame(devrows);rankdev=pd.DataFrame(devrank)
    chosen,status,grid=select_on_dev(dev,rankdev)
    budget_dev=pd.DataFrame(devbudget)
    budget_summary=budget_dev.groupby('method',as_index=False).agg(shots=('total_shots','mean'),size=('mean_set_size','mean'),coverage=('coverage','mean'))
    budget_summary.sort_values(['size','shots','method']).to_csv(out/'development_width_budget_variants.csv',index=False)
    budget_choice=budget_summary.sort_values(['size','shots','method']).iloc[0]['method']
    budget_power=float(budget_choice.replace('width_priority_',''))
    grid.to_csv(out/'development_width_grid.csv',index=False)
    pd.concat([dev,rankdev]).to_csv(out/'development_all_runs.csv',index=False)
    calstr,teststr=chosen.replace('width_cal','').split('_test');ct=float(calstr);tt=float(teststr)
    (out/'frozen_selection.json').write_text(json.dumps({
      'chosen':chosen,'cal_width_threshold':ct,'test_width_threshold':tt,
      'budget_selected_priority':budget_choice,'budget_selected_power':budget_power,
      'status':status,'selection_on':'development seeds 0..4 only',
      'split_policy':SPLIT_POLICY,
      'set_tolerance':TOL_SET,'all_certified_same_alpha':b.ALPHA,
      'all_certified_same_beta':b.BETA,
      'heldout_seed_start':args.eval_start,'heldout_seed_count':args.eval_seeds},indent=2))
    print('SELECTED',chosen,status,'params',ct,tt,'budget priority',budget_choice,flush=True)
    rr=[];details=[]
    for ds in DATASETS:
        for seed in range(args.eval_start,args.eval_start+args.eval_seeds):
            run=init_run(ds,seed,args.epochs)
            rankcal=run['old']['shots'];rankset=run['rankset']
            rr.append(row(run,'rank_V3',rankset,rankcal,run['ranktestshots'],run['qrank'],run['old']['stage'],run['rankstage']))

            q,cs,st=width_cal_threshold(run['yc'],run['cal'],ct)
            s,ts,sttest=width_only_test(q,run['test'],tt,run['rho'])
            rr.append(row(run,'width_selected',s,cs,ts,q,st,sttest,policy=chosen))

            q,cs,st=width_cal_budgeted(run['yc'],run['cal'],rankcal)
            s,ts,sttest=test_with_spending(q,run['test'],run['rho'])
            rr.append(row(run,'width_budget_matched',s,cs,ts,q,st,sttest,
                 policy='greedy widest-first; rank-calibration-budget oracle for analysis'))

            q,cs,st=width_cal_budgeted(run['yc'],run['cal'],rankcal,budget_power)
            s,ts,sttest=test_with_spending(q,run['test'],run['rho'])
            rr.append(row(run,'width_budget_tuned',s,cs,ts,q,st,sttest,
                 policy=f'width / next-increment^{budget_power}; matched budget'))

            s,ts,sttest=width_only_test(run['qrank'],run['test'],tt,run['rho'])
            rr.append(row(run,'width_test_only',s,rankcal,ts,run['qrank'],run['old']['stage'],sttest,
                 policy=f'test_width={tt}'))

            sv,tsv,_=b.test_v2(run['qrank'],run['test'],True)
            rr.append(row(run,'V2_endpoint',sv,rankcal,tsv,run['qrank'],run['old']['stage']))
            sq,shotsq,_=b.ordinary_qcp(run['yc'],run['cal'],run['test'],64)
            rr.append(row(run,'standard_QCP64',sq,len(run['yc'])*64,len(run['yv'])*64,np.nan))


            for label,sets,shot in [('rank_V3',rankset,rankcal+run['ranktestshots']),
                                    ('width_selected',width_only_test(width_cal_threshold(run['yc'],run['cal'],ct)[0],run['test'],tt,run['rho'])[0],None)]:
                for i in range(len(run['yv'])):
                    details.append(dict(dataset=ds,seed=seed,index=i,method=label,
                       label_true=int(run['yv'][i]),set_size=int(sets[i].sum()),covered=int(sets[i,run['yv'][i]])))
            print('EVAL',ds,seed,'rank',rr[-7]['total_shots'],'width',rr[-6]['total_shots'],'tuned',rr[-4]['total_shots'],flush=True)
    hold=pd.DataFrame(rr);hold.to_csv(out/'heldout_all_runs.csv',index=False)
    pd.DataFrame(details).to_csv(out/'heldout_sample_details.csv',index=False)
    sm=hold.groupby('method',as_index=False).agg(mean_shots=('total_shots','mean'),sd_shots=('total_shots','std'),
      mean_cal_shots=('cal_shots','mean'),mean_test_shots=('test_shots','mean'),
      mean_set_size=('mean_set_size','mean'),mean_coverage=('coverage','mean'),
      mean_ideal_containment=('ideal_set_containment','mean'),mean_match_rank=('exact_set_match_rank','mean'))
    sm.to_csv(out/'heldout_summary.csv',index=False)
    by=hold.groupby(['dataset','method'],as_index=False).agg(mean_shots=('total_shots','mean'),
      mean_set_size=('mean_set_size','mean'),coverage=('coverage','mean'),containment=('ideal_set_containment','mean'))
    by.to_csv(out/'heldout_by_dataset.csv',index=False)
    pairs=quality_stats(hold)
    (out/'paired_metrics.json').write_text(json.dumps({'selection':status,'chosen':chosen,'budget_selected_power':budget_power,
      'paired':pairs,'split_policy':SPLIT_POLICY,'runtime_seconds':time.time()-t0,'n_heldout_runs':args.eval_seeds*len(DATASETS),
      'n_model_evaluations':len(hold),'sample_n':int(len(details)/2)},indent=2))
    plots(hold,grid,out)
    print('SUMMARY\n',sm.to_string(index=False));print('PAIRED\n',json.dumps(pairs,indent=2))

if __name__=='__main__':main()


In [ ]:
%%writefile run_factorial.py
import argparse, json, time
from pathlib import Path
import numpy as np, pandas as pd
import experiment_width_matched as exp
import qcp_rank_v2 as base
from v3_reference import test_with_spending

METHODS=['R+D','W+D','R+W','W+W','Wbudget+D']

def run_one(ds, seed, epochs=130):
    run=exp.init_run(ds,seed,epochs)
    yc=run['yc']; sc=run['cal']; st=run['test']; rho=run['rho']
    rank_c=run['old']['shots']; rank_q=run['qrank']
    width_q,width_c,width_stage=exp.width_cal_threshold(yc,sc,0.18)
    budget_q,budget_c,budget_stage=exp.width_cal_budgeted(yc,sc,rank_c,0.)
    sets_rd,td,_=test_with_spending(rank_q,st,rho)
    sets_wd,twd,_=test_with_spending(width_q,st,rho)
    sets_rw,trw,_=exp.width_only_test(rank_q,st,0.12,rho)
    sets_ww,tww,_=exp.width_only_test(width_q,st,0.12,rho)
    sets_bd,tbd,_=test_with_spending(budget_q,st,rho)
    items=[('R+D',sets_rd,rank_c,td,rank_q),('W+D',sets_wd,width_c,twd,width_q),
         ('R+W',sets_rw,rank_c,trw,rank_q),('W+W',sets_ww,width_c,tww,width_q),
         ('Wbudget+D',sets_bd,budget_c,tbd,budget_q)]
    rows=[exp.row(run,name,sets,cs,ts,q) for name,sets,cs,ts,q in items]

    details=[]
    for name,sets,cs,ts,q in items:
        for i in range(len(run['yv'])):
            details.append({'dataset':ds,'seed':seed,'sample_index':i,'method':name,
                'covered':int(sets[i,run['yv'][i]]),'set_size':int(sets[i].sum()),
                'set_mask':int(sets[i,0])+2*int(sets[i,1])})
    return rows,details

def main():
    ap=argparse.ArgumentParser();ap.add_argument('--start',type=int,default=50)
    ap.add_argument('--n-seeds',type=int,default=10);ap.add_argument('--epochs',type=int,default=130)
    ap.add_argument('--split-policy',choices=['random_holdout','original_stratified'],default='random_holdout')
    args=ap.parse_args()
    exp.SPLIT_POLICY=args.split_policy
    root=Path(__file__).parent; out=root/('RESULTS_UNSTRATIFIED' if args.split_policy=='random_holdout' else 'RESULTS_STRATIFIED');out.mkdir(exist_ok=True)
    start=time.time();rows=[];details=[]
    for ds in exp.DATASETS:
        for seed in range(args.start,args.start+args.n_seeds):
            a,b=run_one(ds,seed,args.epochs);rows.extend(a);details.extend(b)
            print(f'completed {ds} {seed}, elapsed {time.time()-start:.1f}s',flush=True)
    raw=pd.DataFrame(rows); det=pd.DataFrame(details)
    raw.to_csv(out/'factorial_all_runs.csv',index=False);det.to_csv(out/'factorial_sample_details.csv',index=False)
    agg=raw.groupby('method',as_index=False).agg(mean_shots=('total_shots','mean'),
        mean_cal_shots=('cal_shots','mean'),mean_test_shots=('test_shots','mean'),
        mean_set_size=('mean_set_size','mean'),mean_coverage=('coverage','mean'),
        mean_ideal_containment=('ideal_set_containment','mean'))
    agg.to_csv(out/'factorial_summary.csv',index=False)
    p=raw.pivot(index=['dataset','seed'],columns='method',values=['total_shots','cal_shots','test_shots','mean_set_size','coverage'])
    def cmp(a,b):
        return dict(a=a,b=b,saving_pct=float(100*(1-p['total_shots'][a]/p['total_shots'][b]).mean()),
          mean_shot_delta=float((p['total_shots'][a]-p['total_shots'][b]).mean()),
          mean_set_delta=float((p['mean_set_size'][a]-p['mean_set_size'][b]).mean()),
          mean_coverage_delta=float((p['coverage'][a]-p['coverage'][b]).mean()))
    contrasts=[cmp('R+D','W+D'),cmp('R+W','W+W'),cmp('R+D','R+W'),cmp('W+D','W+W'),
       cmp('R+D','W+W'),cmp('R+D','Wbudget+D')]

    idx=p.index;grp=[np.flatnonzero(idx.get_level_values(0)==ds) for ds in exp.DATASETS];rng=np.random.default_rng(881300)
    boots=[]
    for _ in range(3000):
        sampled=np.concatenate([rng.choice(grp[g],size=len(grp[g]),replace=True) for g in rng.integers(0,3,size=3)])
        boots.append([100*(1-(p['total_shots'][a].to_numpy()[sampled]/p['total_shots'][b].to_numpy()[sampled])).mean() for a,b in [('R+D','W+D'),('R+D','W+W'),('R+D','Wbudget+D')]])
    bs=np.asarray(boots)
    for pair,dist in zip([('R+D','W+D'),('R+D','W+W'),('R+D','Wbudget+D')],bs.T):
        for r in contrasts:
            if r['a']==pair[0] and r['b']==pair[1]:r['bootstrap_95_pct']=list(np.quantile(dist,[.025,.975]));break

    refpath=Path(__file__).parent/'REFERENCE_heldout_all_runs.csv'
    check={}
    if refpath.exists() and args.split_policy=='random_holdout' and args.start==50:
        ref=pd.read_csv(refpath)
        for mine,old in [('R+D','rank_V3'),('W+W','width_selected'),('R+W','width_test_only'),('Wbudget+D','width_budget_matched')]:
            actual=raw[raw.method==mine].sort_values(['dataset','seed']).reset_index(drop=True)
            expected=ref[(ref.method==old)&(ref.seed.between(args.start,args.start+args.n_seeds-1))].sort_values(['dataset','seed']).reset_index(drop=True)
            comparison={c:float(np.max(np.abs(actual[c].to_numpy()-expected[c].to_numpy()))) for c in ['total_shots','cal_shots','test_shots','mean_set_size','coverage']}
            check[mine]=comparison
            if len(actual)!=len(expected) or any(z>1e-9 for z in comparison.values()):raise AssertionError(('reproduction mismatch',mine,comparison))
    result={'protocol':args.split_policy,'seeds':[args.start,args.start+args.n_seeds-1],
        'alpha':base.ALPHA,'beta':base.BETA,'frozen_thresholds':{'calibration_width':0.18,'test_width':0.12},
        'n_runs':len(raw)//len(METHODS),'contrasts':contrasts,
        'reproduction_checks':check,'wall_seconds':time.time()-start}
    (out/'factorial_metrics.json').write_text(json.dumps(result,indent=2))
    print(agg.to_string(index=False));print(json.dumps(result,indent=2));print('OUTPUT',out,flush=True)
if __name__=='__main__': main()


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ.update({"OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"})
for policy, start in [("random_holdout", 50), ("original_stratified", 40)]:
    subprocess.run([sys.executable, "run_factorial.py", "--split-policy", policy, "--start", str(start), "--n-seeds", "10", "--epochs", "130"], check=True)

import zipfile
with zipfile.ZipFile("QCP_FACTORIAL_RESULTS.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in (Path("RESULTS_UNSTRATIFIED"), Path("RESULTS_STRATIFIED")):
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(path)
print("Saved QCP_FACTORIAL_RESULTS.zip")
try:
    from google.colab import files
    files.download("QCP_FACTORIAL_RESULTS.zip")
except ImportError:
    pass